In [ ]:
# Clone repository and navigate to root
!git clone https://github.com/geetikavasistha-01/Argus.git
%cd Argus

# Argus: Severstal Steel Defect Detection - YOLOv8-seg Training Walkthrough
This notebook provides a complete guide to fine-tuning a YOLOv8-seg model on the Kaggle Severstal Steel Defect Detection dataset, exporting weights, and generating validation metrics.

### 1. Setup Environment & Credentials
First, we install the required packages and configure Kaggle API credentials to download the dataset automatically.

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless pandas numpy scikit-learn matplotlib tqdm pydantic

In [ ]:
# Configure Kaggle credentials
import os
import json
from google.colab import files

kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(kaggle_json_path):
    print("kaggle.json not found in ~/.kaggle. Prompting for upload...")
    uploaded = files.upload()
    if "kaggle.json" in uploaded:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(kaggle_json_path, "wb") as f:
            f.write(uploaded["kaggle.json"])
        os.chmod(kaggle_json_path, 0o600)
        print("kaggle.json configured successfully.")
    else:
        raise FileNotFoundError("kaggle.json was not uploaded.")
else:
    print("kaggle.json already configured.")

### 2. Download and Unzip Dataset
We use the Kaggle CLI to fetch the dataset files.

In [ ]:
# Download the Severstal Steel Defect Detection dataset
!kaggle competitions download -c severstal-steel-defect-detection -p data/severstal

# Verify that the zip file was downloaded successfully
import os
zip_path = "data/severstal/severstal-steel-defect-detection.zip"
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Dataset zip file not found at {zip_path}. "
        "Please verify that the Kaggle API download completed successfully and that "
        "you have accepted the competition rules on the Kaggle website."
    )

# Unzip files
!unzip -q data/severstal/severstal-steel-defect-detection.zip -d data/severstal

### 3. Convert RLE Masks to YOLO Segments
We run the RLE to YOLO conversion script to create class labels `defect_class_1` to `defect_class_4` and set up the directory structure.

In [ ]:
# Convert RLE annotations to YOLO normalized polygons and split dataset 90/10
!python cv/data_prep/rle_to_yolo.py --data-dir data/severstal --output-dir data/severstal_yolo --split 0.1

### 4. Fine-Tune the YOLOv8-seg Model
We train the segmentation model on the prepared dataset. Standard hyperparameters are set for optimal convergence.

In [ ]:
# Fine-tune YOLOv8s-seg on GPU for up to 50 epochs
!python cv/train.py --epochs 50 --imgsz 640 --batch 16 --model-size 8s --save-period 5 --patience 10

### 5. Evaluate the Model & Generate JSON Report
Finally, we run the evaluation script to calculate overall and per-class precision/recall and generate `cv/runs/eval_report.json` for validation.

In [ ]:
# Run evaluation on the validation split
!python cv/evaluate.py --weights models/severstal_yolov8s_seg_best.pt